# 01 — Exploratory Data Analysis: Ahmedabad AQI

**Objective:** Understand the raw data before any modelling — distributions, gaps, seasonality, pollutant correlations, and validate feature engineering output.

**Dataset:** Kaggle — [Air Quality Data in India (2015–2020)](https://www.kaggle.com/datasets/rohanrao/air-quality-data-in-india)  
**File used:** `data/raw/city_day.csv`

In [ ]:
import sys
sys.path.insert(0, '..')  # allow imports from project root when run from notebooks/

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from config import DATA_RAW_DIR, POLLUTANT_COLS

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#999',
    'ytick.color':      '#999',
    'text.color':       '#ccc',
    'grid.color':       '#333',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'figure.titlesize': 14,
})

print('Imports OK')

## 1. Load Raw Data

In [ ]:
raw_path = DATA_RAW_DIR / 'city_day.csv'
raw = pd.read_csv(raw_path)
print(f'Total rows in dataset: {len(raw):,}')
print(f'Cities available: {raw["City"].nunique()}')
print(f'\nColumns: {list(raw.columns)}')

In [ ]:
# Filter Ahmedabad
df = raw[raw['City'] == 'Ahmedabad'].copy()
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').set_index('Date')

print(f'Ahmedabad rows: {len(df)}')
print(f'Date range: {df.index.min().date()} → {df.index.max().date()}')
df.head()

## 2. Missing Value Analysis

In [ ]:
cols_of_interest = [c for c in POLLUTANT_COLS if c in df.columns] + ['AQI']
null_pct = df[cols_of_interest].isna().mean() * 100

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(null_pct.index, null_pct.values, color='#e05c5c', edgecolor='#444')
ax.set_xlabel('Missing %')
ax.set_title('Missing Values by Column — Ahmedabad')
ax.axvline(20, color='#ff9900', linestyle='--', linewidth=1, label='20% threshold')
ax.legend()
for bar, pct in zip(bars, null_pct.values):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{pct:.1f}%', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print(null_pct.to_string())

## 3. AQI Distribution

In [ ]:
aqi = df['AQI'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
axes[0].hist(aqi, bins=40, color='#5b9bd5', edgecolor='#333')
axes[0].axvline(aqi.mean(), color='#ffcc00', linestyle='--', label=f'Mean {aqi.mean():.0f}')
axes[0].axvline(aqi.median(), color='#99ff99', linestyle='--', label=f'Median {aqi.median():.0f}')
axes[0].set_xlabel('AQI')
axes[0].set_ylabel('Days')
axes[0].set_title('AQI Distribution')
axes[0].legend()

# AQI category bands
categories = [
    (0,   50,  '#00e400', 'Good'),
    (51,  100, '#92d050', 'Satisfactory'),
    (101, 200, '#ffff00', 'Moderate'),
    (201, 300, '#ff7e00', 'Poor'),
    (301, 400, '#ff0000', 'Very Poor'),
    (401, 999, '#7e0023', 'Severe'),
]
counts = []
labels = []
colors = []
for lo, hi, color, label in categories:
    n = ((aqi >= lo) & (aqi <= hi)).sum()
    counts.append(n)
    labels.append(f'{label}\n({lo}–{min(hi,500)})')
    colors.append(color)

axes[1].pie(counts, labels=labels, colors=colors,
            autopct=lambda p: f'{p:.1f}%' if p > 2 else '',
            startangle=90, textprops={'fontsize': 8})
axes[1].set_title('Days by AQI Category')

plt.suptitle('Ahmedabad AQI Analysis', y=1.01)
plt.tight_layout()
plt.show()

print(f'\nStats:  min={aqi.min():.0f}  max={aqi.max():.0f}  mean={aqi.mean():.1f}  std={aqi.std():.1f}  skew={aqi.skew():.2f}')

## 4. AQI Time Series

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

# AQI category shading
for lo, hi, color, label in categories:
    ax.axhspan(lo, min(hi, aqi.max() + 20), alpha=0.07, color=color)

ax.plot(df.index, df['AQI'], color='#5b9bd5', linewidth=0.8, alpha=0.9)
ax.plot(df.index, df['AQI'].rolling(30).mean(), color='#ffcc00',
        linewidth=1.5, label='30-day rolling mean')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1, 7]))
plt.xticks(rotation=30)
ax.set_ylabel('AQI')
ax.set_title('Ahmedabad Daily AQI (Full History)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## 5. Seasonality — Monthly & Weekly Patterns

In [ ]:
df_tmp = df[['AQI']].copy().dropna()
df_tmp['month'] = df_tmp.index.month
df_tmp['dow']   = df_tmp.index.dayofweek  # 0=Mon

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
dow_names   = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Monthly boxplot
monthly_data = [df_tmp[df_tmp['month'] == m]['AQI'].values for m in range(1, 13)]
bp1 = axes[0].boxplot(monthly_data, labels=month_names, patch_artist=True,
                      medianprops=dict(color='#ffcc00', linewidth=2))
for patch in bp1['boxes']:
    patch.set_facecolor('#5b9bd5')
    patch.set_alpha(0.7)
axes[0].set_title('AQI by Month')
axes[0].set_ylabel('AQI')
axes[0].grid(True, axis='y')

# Day-of-week boxplot
dow_data = [df_tmp[df_tmp['dow'] == d]['AQI'].values for d in range(7)]
bp2 = axes[1].boxplot(dow_data, labels=dow_names, patch_artist=True,
                      medianprops=dict(color='#ffcc00', linewidth=2))
for patch in bp2['boxes']:
    patch.set_facecolor('#e05c5c')
    patch.set_alpha(0.7)
axes[1].set_title('AQI by Day of Week')
axes[1].set_ylabel('AQI')
axes[1].grid(True, axis='y')

plt.suptitle('Seasonal & Weekly Patterns', y=1.01)
plt.tight_layout()
plt.show()

## 6. Pollutant Correlation Matrix

In [ ]:
corr_cols = [c for c in POLLUTANT_COLS if c in df.columns] + ['AQI']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax,
    annot_kws={'size': 8},
)
ax.set_title('Pollutant Correlation Matrix — Ahmedabad')
plt.tight_layout()
plt.show()

# AQI correlations ranked
print('\nCorrelation with AQI (ranked):')
print(corr['AQI'].drop('AQI').sort_values(ascending=False).to_string())

## 7. Lag Autocorrelation — How Predictive Is Past AQI?

In [ ]:
from pandas.plotting import autocorrelation_plot

lags = list(range(1, 32))
autocorrs = [df['AQI'].dropna().autocorr(lag=lag) for lag in lags]

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(lags, autocorrs, color='#5b9bd5', edgecolor='#333')
ax.axhline(0.2, color='#ff9900', linestyle='--', linewidth=1, label='r=0.2 threshold')
ax.axhline(0,   color='#888',     linestyle='-',  linewidth=0.8)
ax.set_xlabel('Lag (days)')
ax.set_ylabel('Autocorrelation')
ax.set_title('AQI Autocorrelation by Lag (1–31 days)')
ax.set_xticks(lags)
ax.legend()
ax.grid(True, axis='y')
plt.tight_layout()
plt.show()

for lag, ac in zip(lags, autocorrs):
    print(f'  lag {lag:2d}d  →  r = {ac:.3f}')

## 8. Validate Feature Engineering Output

In [ ]:
from pipeline.preprocess import build_features, get_feature_columns

feat_df = build_features()
feature_cols = get_feature_columns(feat_df)

print(f'Rows after processing: {len(feat_df)}')
print(f'Features: {len(feature_cols)}')
print(f'\nFeature list:')
for c in feature_cols:
    print(f'  {c}')

feat_df.head(3)

In [ ]:
# Check for any remaining NaNs — should be 0 after preprocessing
null_check = feat_df.isna().sum()
remaining_nulls = null_check[null_check > 0]

if len(remaining_nulls) == 0:
    print('✓ Zero NaN values in processed dataset — ready for training')
else:
    print('⚠ Remaining NaNs:')
    print(remaining_nulls)

In [ ]:
# Visualise a few engineered features against AQI
lag_cols = [c for c in feature_cols if c.startswith('aqi_lag')]

fig, axes = plt.subplots(1, len(lag_cols), figsize=(14, 4), sharey=True)
for ax, col in zip(axes, lag_cols):
    ax.scatter(feat_df[col], feat_df['AQI'], alpha=0.2, s=5, color='#5b9bd5')
    ax.set_xlabel(col)
    ax.set_title(f'r = {feat_df[col].corr(feat_df["AQI"]):.3f}')
    ax.grid(True)

axes[0].set_ylabel('AQI (target)')
plt.suptitle('Lag Features vs AQI Target', y=1.01)
plt.tight_layout()
plt.show()

## 9. EDA Summary

| Finding | Implication |
|---|---|
| AQI is highest in Nov–Feb (winter) | Add `is_winter` flag ✓ |
| AQI drops sharply June–Sep (monsoon) | Add `is_monsoon` flag ✓ |
| Lag-1d autocorrelation is strongest | `aqi_lag_1d` will be top feature |
| PM2.5 & PM10 most correlated with AQI | Include raw pollutant features |
| Day-of-week effect is weak | Cyclical DOW encoding still kept |
| Some pollutants have >20% missing | Forward-fill short gaps + tolerance |

**Next step → Phase 4:** Train XGBoost on the processed dataset.